# Petrobras Financial Statement Validator

Este projeto tem como objetivo analisar as demonstrações financeiras da Petrobras utilizando dados públicos da CVM.

A proposta é aplicar conceitos de análise de dados, SQL e auditoria contábil para validar a consistência das informações e identificar possíveis anomalias.

## Objetivos

- Validar consistência contábil
- Detectar anomalias trimestrais
- Consultar dados com SQL
- Preparar dados para visualização no Power BI

## 1. Leitura e filtragem dos dados

Nesta etapa, carreguei os dados de demonstrações financeiras da CVM e filtramos apenas os registros da Petrobras.

Também restringi a análise para os anos de 2023 e 2024, tornando o projeto mais objetivo e focado.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/dfp_cia_aberta_DRE_con.csv", sep=";", encoding="latin1")

petrobras = df[df["DENOM_CIA"].str.contains("PETROBRAS", case=False, na=False)]
petrobras = petrobras[petrobras["DT_REFER"].str.contains("2023|2024")]

In [ ]:
### Prévia dos dados filtrados
Abaixo visualizamos uma amostra dos dados da Petrobras após o filtro inicial.

In [ ]:
petrobras[["DT_REFER", "DS_CONTA", "VL_CONTA"]].head()

## 2. Seleção das contas relevantes

Como as demonstrações financeiras possuem muitas contas, selecionei apenas algumas contas-chave para análise:

- Receita líquida
- Lucro líquido
- Caixa e equivalentes
- Empréstimos
- Patrimônio líquido

Essas contas permitem avaliar desempenho, endividamento e estrutura financeira da empresa.

In [ ]:
selected_accounts = [
    "Receita líquida",
    "Lucro líquido",
    "Caixa e equivalentes",
    "Empréstimos",
    "Patrimônio líquido"
]

df_clean = petrobras[petrobras["DS_CONTA"].isin(selected_accounts)].copy()

### Prévia das contas selecionadas
Após selecionar as contas principais, visualizamos a estrutura final utilizada nas análises.

In [ ]:
df_clean.head()

## 3. Padronização dos dados

Para facilitar a análise, renomeei as colunas para nomes mais intuitivos.

In [ ]:
df_clean = df_clean.rename(columns={
    "DT_REFER": "date",
    "DS_CONTA": "account",
    "VL_CONTA": "value"
})

## 4. Armazenamento em banco de dados (SQLite)

Os dados tratados são armazenados em um banco SQLite, permitindo consultas SQL para análise posterior.

In [ ]:
import sqlite3

conn = sqlite3.connect("../data/processed/petrobras.db")
df_clean.to_sql("financial_statements", conn, if_exists="replace", index=False)

## 5. Análise da receita com SQL

Utilizamos SQL para consultar a evolução da receita líquida ao longo do tempo.

Essa abordagem permite explorar os dados de forma estruturada, semelhante ao ambiente corporativo.

In [ ]:
query = """
SELECT DT_REFER, VL_CONTA
FROM financial_statements
WHERE DS_CONTA = 'Receita líquida'
ORDER BY DT_REFER
"""
revenue = pd.read_sql(query, conn)
revenue.head()  

## 6. Cálculo do nível de endividamento (Debt Ratio)

Utilizamos um JOIN em SQL para combinar diferentes contas e calcular o nível de endividamento da empresa.

O indicador utilizado é:

Debt Ratio = Empréstimos / Patrimônio Líquido

Esse indicador ajuda a entender o grau de alavancagem financeira da empresa.

In [ ]:
query = """
SELECT
    a.DT_REFER,
    a.VL_CONTA AS debt,
    b.VL_CONTA AS equity,
    a.VL_CONTA * 1.0 / b.VL_CONTA AS debt_ratio
FROM financial_statements a
JOIN financial_statements b
ON a.DT_REFER = b.DT_REFER
WHERE a.DS_CONTA = 'Empréstimos'
AND b.DS_CONTA = 'Patrimônio líquido'
"""
debt_ratio = pd.read_sql(query, conn)

## 7. Visualização da receita

Abaixo analisamos a evolução da receita líquida ao longo do tempo.

In [ ]:
ax = revenue.plot(
    x="DT_REFER",
    y="VL_CONTA",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Evolução da Receita Líquida - Petrobras"
)

ax.set_xlabel("Período")
ax.set_ylabel("Valor")

In [ ]:
ax = debt_ratio.plot(
    x="DT_REFER",
    y="debt_ratio",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Debt Ratio ao longo do tempo"
)

ax.set_xlabel("Período")
ax.set_ylabel("Debt Ratio")

In [ ]:
avg_debt = debt_ratio["debt_ratio"].mean()
print(f"Debt ratio médio: {avg_debt:.2f}")

Observa-se a variação da receita entre os períodos analisados, permitindo identificar tendências de crescimento ou queda.

## 8. Cálculo de variação trimestral (QoQ)

Para identificar mudanças relevantes, calculamos a variação percentual entre períodos consecutivos.

In [ ]:
revenue["qoq"] = revenue["VL_CONTA"].pct_change()


In [ ]:
max_growth = revenue["qoq"].max()
print(f"Maior crescimento trimestral: {max_growth:.2%}")

## 9. Identificação de anomalias

Consideramos como anomalia variações superiores a 25% entre períodos consecutivos.

Essas variações podem indicar eventos relevantes, mudanças operacionais ou possíveis inconsistências.

alerts = revenue[revenue["qoq"].abs() > 0.25]
alerts

## 10. Detecção estatística de outliers (Z-score)

Além da variação trimestral, aplicamos o Z-score para identificar valores estatisticamente distantes da média.

Essa abordagem é útil para detectar períodos atípicos que podem merecer investigação adicional.

In [ ]:
revenue["zscore"] = (
    revenue["VL_CONTA"] - revenue["VL_CONTA"].mean()
) / revenue["VL_CONTA"].std()
ax = debt_ratio.plot(
    x="DT_REFER",
    y="debt_ratio",
    figsize=(12, 5),
    marker="o",
    grid=True,
    title="Debt Ratio ao longo do tempo"
)

ax.set_xlabel("Período")
ax.set_ylabel("Debt Ratio")

Valores de Z-score elevados em módulo indicam possíveis outliers na série temporal da receita.

## 11. Exportação dos dados para Power BI

Após as análises, exportamos os dados tratados e os alertas identificados para arquivos CSV.

Esses arquivos serão utilizados como fonte no dashboard interativo desenvolvido em Power BI.

In [ ]:
df_clean.to_csv("../data/processed/final_dashboard_data.csv", index=False)
alerts.to_csv("../data/processed/alerts.csv", index=False)

## Encerramento da conexão

Por fim, encerramos a conexão com o banco SQLite.

In [ ]:
conn.close()

## 10. Conclusão

A análise permitiu validar a consistência de dados financeiros e identificar variações relevantes nos indicadores da Petrobras.

O uso combinado de Python, SQL e análise exploratória mostrou-se eficiente para replicar processos comuns em auditoria e análise financeira.

Este projeto demonstra a aplicação prática de análise de dados em um contexto real de negócios.

Com isso, o pipeline do projeto é concluído: dados brutos → tratamento → SQL → análise → alertas → visualização.